# Transform: Cleaning the Data
**Module 3 · outpatient_visits.csv**

---

### What this notebook is for

In the Explore step you found the problems. In this step you fix the ones that matter — and document every decision you make.

Two rules that apply throughout:

1. **The brief is the scope.** Fix what would break the analysis. Leave everything else alone.
2. **Never overwrite the raw file.** All changes are made to a working copy. The raw data stays untouched.

---

### The brief

> Who uses the clinic and how often, centred on repeat visitors and uninsured patients, built for the board and reusable for the county funder report.

**Repeat visitor defined as:** a unique patient who returned to the clinic within one month of their preceding visit.

**Uninsured defined as:** patients with no coverage at all (to be confirmed against the county funder's definition before the final report).

**Unit of analysis:** patients, not visits.

---

### Problems carried forward from Explore

| # | Column | Problem | Affects brief? |
|---|--------|---------|----------------|
| 1 | visit_date | Stored as object, not parsed as date | Yes — repeat visitor calculation requires date arithmetic |
| 2 | insurance_type | Possible case or whitespace inconsistencies | Yes — uninsured count depends on clean grouping |
| 3 | visit_type | Possible case or whitespace inconsistencies | Yes — utilisation breakdowns and the care coordinator case depend on this column grouping correctly |
| 4 | county | 248 nulls (30% of rows) | Yes — county funder needs geographic cut |
| 5 | age | 33 nulls; possible out-of-range values | Partial — not a primary metric but may appear in subgroup breakdowns |
| 6 | copay_amount | 44 nulls; possible negatives | Partial — not a primary metric |
| 7 | Duplicates | visit_id and patient_id + visit_date unchecked | Yes — duplicate rows would inflate visit and patient counts |

**Before you start.** Choose **File → Save a copy in Drive** so your work is saved. Open your AI chat and paste the ground-rules prompt from the Explore page of Module 3. If anything stops working: **Runtime → Restart session and run all**.

---
## Section 0 — Setup

Load the raw data. Do not modify anything yet. Confirm the shape matches what you found in Explore (812 rows, 13 columns).

In [ ]:
import pandas as pd
import numpy as np

# Load the raw file — do not modify this dataframe
raw = pd.read_csv('https://raw.githubusercontent.com/EDC-AICC/ai-ds-development/main/src/data/outpatient_visits.csv')

# Working copy — all cleaning happens here
df = raw.copy()

print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')

**Checkpoint:** Does the shape match your Explore output? If not, stop and check that you are loading the correct file.

---
## Section 1 — visit_date: Parse Date Column

**Finding from Explore:** `visit_date` is stored as object (string). It has not been parsed as a date.

**Why it matters for the brief:** The repeat visitor calculation requires date arithmetic — specifically, whether a patient's next visit falls within 30 days of their previous one. You cannot do that calculation on a string column.

---

**Before you write any code, answer these questions:**

1. What will you do with records where the date cannot be parsed (`errors='coerce'` produces NaT)?
2. Will you keep the original string column, or replace it?
3. How will you verify the fix worked?

**Your decision (fill this in before writing code):**

*Write your answers here.*

In [ ]:
# Section 1 — Your cleaning code here
# Parse visit_date to datetime
# Do not modify the raw dataframe


In [ ]:
# Section 1 — Verification
print('dtype:', df['visit_date'].dtype)
print('Date range:', df['visit_date'].min(), 'to', df['visit_date'].max())
print('Unparseable dates (NaT):', df['visit_date'].isna().sum())

**Log entry for Section 1** — complete this before moving on:

| Field | Your entry |
|-------|------------|
| Column | visit_date |
| Problem | Stored as object, not parsed as date |
| Decision | *what you decided* |
| Action taken | *what the code does* |
| Rows affected | *how many* |
| Verified? | Yes / No |

---
## Section 2 — insurance_type: Standardise Values

**Finding from Explore:** `insurance_type` may contain values that differ only by case or whitespace (e.g. `Uninsured` and `uninsured` as separate entries). If so, they will be counted as two groups.

**Why it matters for the brief:** The entire uninsured patient count depends on this column grouping correctly. A case inconsistency doubles — or splits — the group you are reporting on.

---

**Before you write any code, answer these questions:**

1. What case standard will you apply (lower, upper, title)?
2. How will you handle leading or trailing whitespace?
3. Will you check what distinct values remain after standardisation?

**Your decision:**

*Write your answers here.*

In [ ]:
# Section 2 — Your cleaning code here
# Show distinct values BEFORE, standardise, show distinct values AFTER


In [ ]:
# Section 2 — Verification
print(df['insurance_type'].value_counts())

**Log entry for Section 2:**

| Field | Your entry |
|-------|------------|
| Column | insurance_type |
| Problem | Possible case/whitespace inconsistencies |
| Decision | *what you decided* |
| Action taken | *what the code does* |
| Rows affected | *how many* |
| Verified? | Yes / No |

---
## Section 3 — visit_type: Standardise Values

**Finding from Explore:** `visit_type` may contain values that differ only by case or whitespace — the same issue as `insurance_type`.

**Why it matters for the brief:** This column supports two brief requirements. First, the care coordinator hiring decision depends on understanding what repeat visitors are coming back for — if repeat visits are concentrated in a particular visit type, that shapes what kind of coordinator is needed and whether one is the right intervention at all. Second, the county funder will want to see how the clinic's capacity is being used and by which patient groups. A case inconsistency in `visit_type` fragments both of those views.

---

**Before you write any code, answer these questions:**

1. What distinct values does `visit_type` have? Are there any that look like duplicates under a different case or with extra whitespace?
2. What case standard will you apply — and is it consistent with what you chose for `insurance_type`?
3. How will you verify the fix worked?

**Your decision:**

*Write your answers here.*

In [ ]:
# Section 3 — Your cleaning code here
# Show distinct values BEFORE, standardise, show distinct values AFTER


In [ ]:
# Section 3 — Verification
print(df['visit_type'].value_counts())

**Log entry for Section 3:**

| Field | Your entry |
|-------|------------|
| Column | visit_type |
| Problem | Possible case/whitespace inconsistencies |
| Decision | *what you decided* |
| Action taken | *what the code does* |
| Rows affected | *how many* |
| Verified? | Yes / No |

---
## Section 4 — county: Document and Flag Missing Values

**Finding from Explore:** `county` has 248 missing values — 30% of the dataset. The breakdown by insurance_type and visit_type may tell you whether the missingness is concentrated in a particular group.

**Why it matters for the brief:** The county funder will want a geographic cut of the data. If 30% of county values are missing, that cut is incomplete — and if missing county is concentrated among uninsured patients, the most important slice for the funder is also the least complete.

**The constraint:** You cannot impute or invent county. You do not have a secondary source. Dropping rows is also not appropriate — you would lose 30% of your data and introduce bias.

---

**Before you write any code, answer these questions:**

1. What will you do with the missing values? (Options: leave as null and document; add a flag column `county_known`; both)
2. Is the missingness concentrated in a particular insurance group or visit type? Check before deciding.
3. What limitation statement will you include in any deliverable that shows a county breakdown?

**Your decision:**

*Write your answers here.*

In [ ]:
# Section 4 — Your code here
# Check missingness breakdown by insurance_type and visit_type
# Add a flag column if that is your decision
# Do not drop rows. Do not impute.


In [ ]:
# Section 4 — Verification
print('County nulls remaining:', df['county'].isna().sum())
print('Columns in df:', [c for c in df.columns if 'county' in c])

**Log entry for Section 4:**

| Field | Your entry |
|-------|------------|
| Column | county |
| Problem | 248 missing values (30%) |
| Decision | *what you decided — and what you ruled out and why* |
| Action taken | *what the code does* |
| Rows affected | *how many* |
| Limitation to document | *what the deliverable must say about county completeness* |
| Verified? | Yes / No |

---
## Section 5 — age and copay_amount: Out-of-Range Values

**Finding from Explore:**
- `age`: 33 nulls. Any values below 0 or above 110 are implausible.
- `copay_amount`: 44 nulls. Negative values are implausible.

**Why it matters for the brief:** Neither column is a primary metric, but both may appear in subgroup breakdowns. An implausible value in a summary statistic would undermine the deliverable's credibility.

---

**Before you write any code, answer these questions:**

1. For out-of-range age values: will you set them to null, flag them, or drop the row?
2. For negative copay values: same question.
3. For the nulls that already exist: will you leave them, or do something else?
4. What threshold defines out-of-range for age? For copay?

**Your decision:**

*Write your answers here.*

In [ ]:
# Section 5 — Your cleaning code here
# Handle age out-of-range values
# Handle copay_amount negatives
# Check null counts before and after


In [ ]:
# Section 5 — Verification
for col in ['age', 'copay_amount']:
    print(f'{col}: min={df[col].min()}, max={df[col].max()}, nulls={df[col].isna().sum()}')

**Log entry for Section 5:**

| Field | Your entry |
|-------|------------|
| Columns | age, copay_amount |
| Problem | Out-of-range values; existing nulls |
| Decision (age) | *what you decided* |
| Decision (copay) | *what you decided* |
| Action taken | *what the code does* |
| Rows affected | *how many* |
| Verified? | Yes / No |

---
## Section 6 — Duplicates

**Finding from Explore:** Duplicate records were flagged for investigation — specifically `visit_id` appearing more than once, and `patient_id` + `visit_date` appearing more than once.

**Why it matters for the brief:** A duplicate visit row would inflate both visit counts and — if not caught — patient counts. The repeat visitor calculation in particular depends on accurate visit sequences per patient.

---

**Before you write any code, answer these questions:**

1. What constitutes a true duplicate here — identical rows, or same key with different values?
2. If `visit_id` appears twice with identical data in both rows, which row do you keep?
3. If `visit_id` appears twice with different data, what do you do?
4. How will you confirm the deduplication did not remove rows it should have kept?

**Your decision:**

*Write your answers here.*

In [ ]:
# Section 6 — Your cleaning code here
# Check for duplicates first — show what you find before removing anything
# Then apply your decision


In [ ]:
# Section 6 — Verification
print('Rows before deduplication:', len(raw))
print('Rows after deduplication:', len(df))
print('visit_id duplicates remaining:', df.duplicated(subset='visit_id').sum())

**Log entry for Section 6:**

| Field | Your entry |
|-------|------------|
| Column(s) | visit_id; patient_id + visit_date |
| Problem | Possible duplicate records |
| What you found | *how many duplicates, and of what type* |
| Decision | *what you decided* |
| Action taken | *what the code does* |
| Rows removed | *how many* |
| Verified? | Yes / No |

---
## Section 7 — Cleaning Log

Compile all log entries into one summary. This is the record of every decision made. It travels with the clean dataset.

A cleaning log serves two purposes:
- **Reproducibility:** someone else can follow the same steps and arrive at the same dataset.
- **Accountability:** when someone questions a number in the final report, you can trace it back to a documented decision — not a forgotten choice made six weeks earlier.

Complete the table below.

### Cleaning Log — outpatient_visits.csv

| Section | Column | Problem | Decision | Action | Rows affected | Limitation to document |
|---------|--------|---------|----------|--------|---------------|------------------------|
| 1 | visit_date | | | | | |
| 2 | insurance_type | | | | | |
| 3 | visit_type | | | | | |
| 4 | county | | | | | |
| 5 | age | | | | | |
| 5 | copay_amount | | | | | |
| 6 | visit_id / duplicates | | | | | |

---
## Section 8 — Save the Clean Dataset

Export to a new file. Do not overwrite `outpatient_visits.csv`.

Print a before/after comparison so you have a clear record of what changed.

In [ ]:
# Save the clean dataset
output_path = 'outpatient_visits_clean.csv'
df.to_csv(output_path, index=False)

# Before / after summary
print('=== Before / After Summary ===')
print(f'Rows:    {len(raw)} → {len(df)}')
print(f'Columns: {raw.shape[1]} → {df.shape[1]}')
print()

print('Null counts — key columns:')
cols = ['visit_date', 'insurance_type', 'visit_type', 'county', 'age', 'copay_amount']
for col in cols:
    before = raw[col].isna().sum() if col in raw.columns else 'n/a'
    after  = df[col].isna().sum()  if col in df.columns  else 'n/a'
    print(f'  {col}: {before} → {after}')

print()
print(f'Clean file saved to: {output_path}')

---
## Final Check

Before you close this notebook, answer these three questions:

1. **Did you fix anything the brief does not require?** If yes, undo it.
2. **Is every decision in the cleaning log?** If not, add it now.
3. **Does the clean dataset have the same number of rows as the raw dataset — or fewer, and do you know why?**

The clean dataset and this notebook go together. One without the other is incomplete.